<a href="https://colab.research.google.com/github/rdelhibabu/HQCC_QaaS/blob/main/HQCC_QaaS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
!pip install qiskit qiskit-ibm-runtime qiskit-optimization qiskit-algorithms networkx matplotlib scipy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 327.8/327.8 kB 7.5 MB/s eta 0:00:00


In [4]:
import networkx as nx
import numpy as np
from qiskit.circuit.library import QAOAAnsatz
from qiskit.quantum_info import SparsePauliOp
from qiskit.primitives import StatevectorEstimator
from qiskit_algorithms.optimizers import SPSA

# Step 1: Define the Graph (N=4 for a quick local test)
num_nodes = 4
G = nx.random_regular_graph(d=2, n=num_nodes)

# Step 2: Construct the Ising Hamiltonian (Cost Hamiltonian H_c)
def build_max_cut_hamiltonian(graph):
    pauli_list = []
    for edge in graph.edges():
        i, j = edge
        # Create Z_i * Z_j terms
        pauli_string = ['I'] * len(graph.nodes())
        pauli_string[i] = 'Z'
        pauli_string[j] = 'Z'
        # Qiskit uses little-endian ordering, so we reverse the string
        pauli_list.append(("".join(pauli_string)[::-1], 0.5))

    return SparsePauliOp.from_list(pauli_list)

H_c = build_max_cut_hamiltonian(G)

# Step 3: Construct the Parameterized QAOA Ansatz
depth_p = 1
qaoa_circuit = QAOAAnsatz(cost_operator=H_c, reps=depth_p)

# Step 4: Define the V2 Estimator and SPSA Optimizer
estimator = StatevectorEstimator()
spsa = SPSA(maxiter=85)

# Step 5: Optimization Loop
def cost_function(params):
    # In V2, we pass a list of tuples: (circuit, observable, parameters)
    job = estimator.run([(qaoa_circuit, H_c, params)])

    # Extract the expected value from the V2 result object
    result = job.result()[0]
    expected_value = result.data.evs

    # Minimizing the negative expectation value for Max-Cut
    return -expected_value

# Initialize random parameters
initial_params = 2 * np.pi * np.random.rand(qaoa_circuit.num_parameters)

print(f"Starting QAOA Optimization with SPSA (Depth p={depth_p})...")
result = spsa.minimize(fun=cost_function, x0=initial_params)

print("\nOptimization Complete.")
print(f"Optimal Parameters (Gamma, Beta): {result.x}")
print(f"Optimized Ground State Energy Estimate: {-result.fun}")

Starting QAOA Optimization with SPSA (Depth p=1)...


/usr/local/lib/python3.12/dist-packages/scipy/sparse/linalg/_dsolve/linsolve.py:606: SparseEfficiencyWarning: splu converted its input to CSC format
  return splu(A).solve
/usr/local/lib/python3.12/dist-packages/scipy/sparse/linalg/_matfuncs.py:707: SparseEfficiencyWarning: spsolve is more efficient when sparse b is in the CSC matrix format
  return spsolve(Q, P)
/usr/local/lib/python3.12/dist-packages/scipy/sparse/_index.py:168: SparseEfficiencyWarning: Changing the sparsity structure of a csr_matrix is expensive. lil and dok are more efficient.
  self._set_intXint(row, col, x.flat[0])
/usr/local/lib/python3.12/dist-packages/scipy/sparse/linalg/_dsolve/linsolve.py:606: SparseEfficiencyWarning: splu converted its input to CSC format
  return splu(A).solve
/usr/local/lib/python3.12/dist-packages/scipy/sparse/linalg/_matfuncs.py:707: SparseEfficiencyWarning: spsolve is more efficient when sparse b is in the CSC matrix format
  return spsolve(Q, P)
/usr/local/lib/python3.12/dist-packages/


Optimization Complete.
Optimal Parameters (Gamma, Beta): [6.6758844 7.0685836]
Optimized Ground State Energy Estimate: 0.9999999999999603


In [8]:
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import EstimatorV2 as Estimator
from qiskit_ibm_runtime.options import EstimatorOptions
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager

print("Configuring QaaS Session with ZNE Error Mitigation...")

# In V2, we only pass Resilience Options to the Estimator
options = EstimatorOptions()
options.resilience_level = 2   # Level 2 inherently triggers Zero-Noise Extrapolation (ZNE)

# To run a session against actual hardware:
'''
# 1. Authenticate (Insert your token here)
service = QiskitRuntimeService(channel="ibm_quantum", token="YOUR_TOKEN_HERE")

# 2. Select the backend
backend = service.backend("ibm_brisbane")

# 3. CRITICAL V2 STEP: Transpile the circuit and observable for the specific backend
pass_manager = generate_preset_pass_manager(optimization_level=3, backend=backend)
isa_circuit = pass_manager.run(qaoa_circuit)
isa_observable = H_c.apply_layout(isa_circuit.layout)

# 4. Execute the SPSA loop inside a dedicated Session
with Session(service=service, backend=backend) as session:
    estimator = Estimator(session=session, options=options)

    def cloud_cost_function(params):
        # Pass the transpiled (ISA) circuit and observable
        job = estimator.run([(isa_circuit, isa_observable, params)])
        return -job.result()[0].data.evs

    print("Running mitigated optimization loop on QPU...")
    result = spsa.minimize(fun=cloud_cost_function, x0=initial_params)

    print(f"Mitigated Optimal Energy: {-result.fun}")
    print(f"Optimal Parameters: {result.x}")
'''
print("Run the session block once your IBM API token and backend are configured.")

Configuring QaaS Session with ZNE Error Mitigation...
Run the session block once your IBM API token and backend are configured.


In [10]:
import numpy as np

def parameter_shift_gradient(estimator, circuit, hamiltonian, params, shift=np.pi/2):
    gradients = np.zeros_like(params)

    for i in range(len(params)):
        # Shift parameter forward
        params_plus = np.copy(params)
        params_plus[i] += shift

        # Shift parameter backward
        params_minus = np.copy(params)
        params_minus[i] -= shift

        # Evaluate shifted energies via the V2 Estimator (using the tuple format)
        job_plus = estimator.run([(circuit, hamiltonian, params_plus)])
        job_minus = estimator.run([(circuit, hamiltonian, params_minus)])

        # Extract the expectation values using the V2 result structure
        E_plus = job_plus.result()[0].data.evs
        E_minus = job_minus.result()[0].data.evs

        # Calculate partial derivative
        gradients[i] = 0.5 * (E_plus - E_minus)

    return gradients

# Example calculation using the previous QAOA ansatz parameters
grads = parameter_shift_gradient(estimator, qaoa_circuit, H_c, initial_params)
print("Calculated Parameter-Shift Gradients:", grads)

Calculated Parameter-Shift Gradients: [-8.32667268e-17  8.32667268e-17]


/usr/local/lib/python3.12/dist-packages/scipy/sparse/linalg/_dsolve/linsolve.py:606: SparseEfficiencyWarning: splu converted its input to CSC format
  return splu(A).solve
/usr/local/lib/python3.12/dist-packages/scipy/sparse/linalg/_matfuncs.py:707: SparseEfficiencyWarning: spsolve is more efficient when sparse b is in the CSC matrix format
  return spsolve(Q, P)
/usr/local/lib/python3.12/dist-packages/scipy/sparse/_index.py:168: SparseEfficiencyWarning: Changing the sparsity structure of a csr_matrix is expensive. lil and dok are more efficient.
  self._set_intXint(row, col, x.flat[0])
